# Member 4 — EDA & Statistical Analysis
**CS412 · UK Road Accidents (2024–2025)** — uses the cleaned file from Member 3.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Road_Accident_Project/UK_Road_Accidents_Cleaned_2024_2025.csv',
                 low_memory=False)

df['speed_limit'] = df['speed_limit'].replace(-1, np.nan)   # -1 = unknown
print(df.shape)

## 2. Descriptive statistics (IV.B)

In [ ]:
cols = ['number_of_vehicles', 'speed_limit', 'hour']
summary = pd.DataFrame({
    'Mean': df[cols].mean(),
    'Median': df[cols].median(),
    'SD': df[cols].std(),
    'Min': df[cols].min(),
    'Max': df[cols].max(),
}).round(2)
summary

In [ ]:
# Frequency tables (%)
for c in ['severity', 'light_group', 'weather_group', 'surface_group', 'area']:
    print(df[c].value_counts(normalize=True).mul(100).round(2), '\n')

## 3. Outliers (IV.B)

In [ ]:
q1 = df['number_of_vehicles'].quantile(0.25)
q3 = df['number_of_vehicles'].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
outliers = df[df['number_of_vehicles'] > upper]

print('Upper limit:', upper)
print('Number of outliers:', len(outliers), f'({len(outliers) / len(df) * 100:.2f}%)')

plt.boxplot(df['number_of_vehicles'], vert=False)
plt.title('Number of vehicles per collision')
plt.show()

## 4. Severity by year — Fig. 1 (IV.C)

In [ ]:
fig1 = pd.crosstab(df['severity'], df['collision_year'], normalize='columns') * 100
fig1 = fig1.loc[['Fatal', 'Serious', 'Slight']]

fig1.plot(kind='bar', color=['#7A9CC6', '#2F5D8A'])
plt.title('Collision severity by year, 2024–2025')
plt.ylabel('% of collisions'); plt.xlabel(''); plt.xticks(rotation=0)
plt.legend(title='Year')
plt.savefig('fig1_severity_by_year.png', dpi=300, bbox_inches='tight')
plt.show()
fig1.round(2)

## 5. Hypothesis tests — Table III (IV.G), α = 0.05

In [ ]:
def z_test(a, b):
    p1, p2 = a.mean(), b.mean()
    p = (a.sum() + b.sum()) / (len(a) + len(b))
    z = (p1 - p2) / np.sqrt(p * (1 - p) * (1 / len(a) + 1 / len(b)))
    return z, 2 * stats.norm.sf(abs(z))

def chi_test(col):
    table = pd.crosstab(df[col], df['KSI'])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    return chi2, p, v

y24, y25 = df[df['collision_year'] == 2024], df[df['collision_year'] == 2025]
rural = df[df['area'].str.contains('rural', case=False, na=False)]
urban = df[df['area'].str.contains('urban', case=False, na=False)]
ksi, slight = df[df['KSI'] == 1], df[df['KSI'] == 0]
rows = []

# H1: KSI rate 2024 vs 2025
z, p = z_test(y25['KSI'], y24['KSI'])
rows.append(['H1', f'z = {z:.2f}', p, f"{y24['KSI'].mean()*100:.2f}% vs {y25['KSI'].mean()*100:.2f}%"])
z, p = z_test(y25['KSI_adjusted'], y24['KSI_adjusted'])
rows.append(['H1 (adjusted)', f'z = {z:.2f}', p,
             f"{y24['KSI_adjusted'].mean()*100:.2f}% vs {y25['KSI_adjusted'].mean()*100:.2f}%"])

# H2, H3: light, weather, surface
for h, col in [('H2', 'light_group'), ('H3 weather', 'weather_group'), ('H3 surface', 'surface_group')]:
    chi2, p, v = chi_test(col)
    rows.append([h, f'chi2 = {chi2:.1f}', p, f'V = {v:.3f}'])

# H4: rural > urban
z, p = z_test(rural['KSI'], urban['KSI'])
rows.append(['H4', f'z = {z:.2f}', stats.norm.sf(z),
             f"{rural['KSI'].mean()*100:.2f}% vs {urban['KSI'].mean()*100:.2f}%"])

# H5: higher speed limits in KSI collisions
U, p = stats.mannwhitneyu(ksi['speed_limit'].dropna(), slight['speed_limit'].dropna(), alternative='greater')
rows.append(['H5', f'U = {U:.3g}', p,
             f"mean {ksi['speed_limit'].mean():.1f} vs {slight['speed_limit'].mean():.1f} mph"])

# H6: hour, weekday
for h, col in [('H6 hour', 'hour'), ('H6 weekday', 'weekday')]:
    chi2, p, v = chi_test(col)
    rows.append([h, f'chi2 = {chi2:.1f}', p, f'V = {v:.3f}'])

# H7: vehicles by KSI
U, p = stats.mannwhitneyu(ksi['number_of_vehicles'], slight['number_of_vehicles'])
rows.append(['H7', f'U = {U:.3g}', p,
             f"mean {ksi['number_of_vehicles'].mean():.2f} vs {slight['number_of_vehicles'].mean():.2f}"])

table3 = pd.DataFrame(rows, columns=['H', 'Statistic', 'p', 'Effect'])
table3['Result'] = np.where(table3['p'] < 0.05, 'Reject H0', 'Fail to reject H0')
table3['p'] = table3['p'].map(lambda x: '< 0.001' if x < 0.001 else f'{x:.3f}')
table3

## 6. Numbers for the report

In [ ]:
s = summary
print('IV.B')
print(f"  Vehicles: mean {s.loc['number_of_vehicles','Mean']}, median {s.loc['number_of_vehicles','Median']}, "
      f"SD {s.loc['number_of_vehicles','SD']}, max {s.loc['number_of_vehicles','Max']}")
print(f"  Speed limit: mean {s.loc['speed_limit','Mean']}, median {s.loc['speed_limit','Median']}, SD {s.loc['speed_limit','SD']}")
print(f"  Hour: mean {s.loc['hour','Mean']}, median {s.loc['hour','Median']}, SD {s.loc['hour','SD']}")
for c in ['light_group', 'weather_group', 'surface_group', 'area']:
    top = df[c].value_counts(normalize=True)
    print(f"  Most common {c}: {top.index[0]} ({top.iloc[0]*100:.1f}%)")
print(f"  Outliers: {len(outliers)} collisions ({len(outliers)/len(df)*100:.2f}%) with more than {int(upper)} vehicles")

v_rows = table3[table3['Effect'].str.startswith('V =')].copy()
v_rows['v'] = v_rows['Effect'].str[4:].astype(float)
v_rows = v_rows.sort_values('v', ascending=False)
print('\nStrongest factors (Cramér\'s V):')
print(v_rows[['H', 'Effect']].to_string(index=False))

## 7. Findings and Conclusion
Written in the report (sections IV.H and V).